# 02 — Standardization

**Tujuan:** mengurangi perbedaan FORMAT (bukan perbedaan identitas) berdasarkan temuan aktual Notebook 01.

**Yang mendasari desain di notebook ini (fakta dari Notebook 01, bukan asumsi):**
- `phone_number`: 91.8% mengandung simbol format; sebagian mengandung extension pola `...xNNNNN` yang harus dipisah dulu sebelum membandingkan digit.
- `first_name`/`last_name`: whitespace (2.634/2.651 baris), kapitalisasi tidak konsisten (562/855 grup), dan sisipan karakter acak (`Mic2hael`, `AAnnttonio`) — TIDAK boleh dibersihkan agresif (mis. menghapus semua digit dari nama) karena berisiko mengubah identitas asli; fokus hanya pada normalisasi format (lowercase, whitespace) untuk keperluan MATCHING, bukan mengoreksi ejaan.
- `email`: formatnya sudah bersih (0 invalid/uppercase/spasi) — standardisasi minimal (lowercase, trim) saja, tanpa aturan provider-specific.
- `country`: 243 unique value, indikasi variasi penulisan — akan dicek pola aktual sebelum dibuat mapping (jangan mengarang daftar sinonim negara tanpa lihat datanya).
- `address`: 498 baris ALL CAPS — normalisasi ringan (lowercase, whitespace), tidak menghapus informasi.

**Aturan yang berlaku (Rule 3 — Preserve raw data):**
- `df_raw` (hasil load dari CSV) TIDAK PERNAH diubah di notebook ini.
- Semua transformasi menghasilkan kolom baru dengan suffix `_std` di dataframe terpisah `df_std`.
- Output notebook ini: `customers_standardized.csv` (file baru, bukan overwrite raw), disimpan langsung di `/content` (tanpa subfolder, sesuai kesepakatan).

In [31]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

RAW_PATH = r"C:\Users\User\Downloads\Fix\data\raw\crm_50000_customers_dirty_v3.csv"
OUTPUT_PATH = r"C:\Users\User\Downloads\Fix\data\raw\customers_standarized.csv"


In [32]:
df_raw = pd.read_csv(RAW_PATH, sep=";", encoding="utf-8")
print(f"Raw loaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} cols")

# df_std adalah copy eksplisit -- df_raw tidak pernah disentuh setelah ini
df_std = df_raw.copy()


Raw loaded: 50,000 rows x 14 cols


## 1. Name standardization (`first_name`, `last_name`)

**Transformasi yang dilakukan (ringan, non-destruktif):**
- lowercase
- trim whitespace + normalize repeated whitespace

**Sengaja TIDAK dilakukan di sini:**
- Menghapus digit/simbol yang tersisip (`Mic2hael` -> `Michael`) — itu koreksi typo, bukan standardisasi format, dan berisiko salah tebak. Fuzzy comparison di Splink (Notebook 04) yang akan menangani ini, bukan aturan hardcode di sini.

In [33]:
def normalize_text_basic(series):
    s = series.astype(str)
    s = s.str.lower()
    s = s.str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    # kembalikan NaN asli (astype(str) mengubah NaN jadi string 'nan')
    s = s.mask(series.isna(), np.nan)
    return s

df_std["first_name_std"] = normalize_text_basic(df_std["first_name"])
df_std["last_name_std"] = normalize_text_basic(df_std["last_name"])

df_std[["first_name", "first_name_std", "last_name", "last_name_std"]].sample(10, random_state=42)


,first_name,first_name_std,last_name,last_name_std
33553,Nicole,nicole,Potts,potts
9427,Jessica*,jessica*,HHaanncock,hhaanncock
199,Kimberly,kimberly,Cox!,cox!
12447,Danielle,danielle,Johnson,johnson
39489,Darren,darren,Rogers,rogers
42724,Diane,diane,Wilkins,wilkins
10822,Valerie,valerie,Walker,walker
49498,Carla,carla,Acevedo,acevedo
4144,Dylan,dylan,Manning,manning
36958,LINDA,linda,ANDRADE,andrade


## 2. Email standardization

Format sudah bersih dari Notebook 01 (0 invalid/uppercase/spasi) — standardisasi minimal saja.

**Catatan dari Notebook 01 yang TIDAK diselesaikan di sini:** ditemukan pola email `sharedNNNN@...` dan email yang berisi nama orang berbeda dari `first_name`/`last_name` di baris yang sama. Ini BUKAN masalah format, jadi tidak diperbaiki di standardization — akan jadi catatan analisis di Notebook 03/04 (email tidak boleh jadi sinyal match tunggal).

In [34]:
df_std["email_std"] = normalize_text_basic(df_std["email"])

# Deteksi (bukan diperbaiki) pola email 'shared' untuk dicatat sebagai flag analisis
df_std["email_is_shared_pattern"] = df_std["email_std"].str.contains(r"^shared\d+@", regex=True, na=False)
print(f"Baris dengan pola email 'sharedNNNN@...': {df_std['email_is_shared_pattern'].sum()}")

df_std[["email", "email_std", "email_is_shared_pattern"]].sample(10, random_state=42)


Baris dengan pola email 'sharedNNNN@...': 1550


,email,email_std,email_is_shared_pattern
33553,nicole.potts659@gmail.com,nicole.potts659@gmail.com,False
9427,jessica.hancock630@gmail.com,jessica.hancock630@gmail.com,False
199,kimberly.cox676@yahoo.com,kimberly.cox676@yahoo.com,False
12447,danielle.johnson048@yahoo.com,danielle.johnson048@yahoo.com,False
39489,darren.rogers723@yahoo.com,darren.rogers723@yahoo.com,False
42724,diane.wilkins421@yahoo.com,diane.wilkins421@yahoo.com,False
10822,valerie.walker476@yahoo.com,valerie.walker476@yahoo.com,False
49498,carla.acevedo938@hotmail.com,carla.acevedo938@hotmail.com,False
4144,dylan.manning784@yahoo.com,dylan.manning784@yahoo.com,False
36958,linda.andrade601@gmail.com,linda.andrade601@gmail.com,False


## 3. Phone standardization

**Langkah wajib berdasarkan temuan Notebook 01:** pisahkan extension (`...xNNNNN`) SEBELUM menghitung/membandingkan digit utama. Tidak mengarang country code — jika ada prefix `+1` atau `001` dipertahankan apa adanya sebagai bagian dari digit, tidak diasumsikan semua nomor US.

In [35]:
def split_phone_extension(series):
    s = series.astype(str)
    # pola extension: 'x' diikuti digit, di akhir string (case-insensitive)
    ext_pattern = re.compile(r"x(\d+)\s*$", flags=re.IGNORECASE)

    extensions = s.str.extract(ext_pattern, expand=False)
    main_part = s.str.replace(ext_pattern, "", regex=True)

    # bagian utama: buang semua karakter non-digit
    main_digits = main_part.str.replace(r"\D", "", regex=True)

    main_digits = main_digits.mask(series.isna(), np.nan)
    extensions = extensions.where(series.notna(), np.nan)
    return main_digits, extensions

df_std["phone_main_std"], df_std["phone_extension_std"] = split_phone_extension(df_std["phone_number"])

print("Distribusi panjang digit phone_main_std SETELAH extension dipisah:")
print(df_std["phone_main_std"].dropna().str.len().value_counts().sort_index())


Distribusi panjang digit phone_main_std SETELAH extension dipisah:
phone_main_std
3        36
4      1808
5       212
7         1
8        83
9       744
10    33246
11     5964
13     7906
Name: count, dtype: int64


In [36]:
df_std[["phone_number", "phone_main_std", "phone_extension_std"]].sample(10, random_state=42)


,phone_number,phone_main_std,phone_extension_std
33553,001-640-729-1553x3879,0016407291553,3879
9427,2.306.799.769,2306799769,NaN
199,087-265-6604,0872656604,NaN
12447,5.530.801.979,5530801979,NaN
39489,(264)226-4218,2642264218,NaN
42724,001-272-370-9902x682,0012723709902,682
10822,+1-897-770-1091x7231,18977701091,7231
49498,+1-370-645-8796x99924,13706458796,99924
4144,+1-413-997-4858x0456,14139974858,0456
36958,448-698-2727x632,4486982727,632


**Cek hasil di atas:** apakah distribusi panjang digit sekarang jauh lebih rapat (mis. mayoritas 10-11 digit) dibanding sebelum extension dipisah (10-18 digit)? Ini validasi langsung bahwa hipotesis extension di Notebook 01 benar. Jika distribusi MASIH lebar setelah ini, regex `ext_pattern` perlu direvisi (kemungkinan ada pola extension lain yang belum tertangkap) — jangan lanjut ke Notebook 03 sebelum ini rapi.

## 4. Address standardization

In [37]:
df_std["address_std"] = normalize_text_basic(df_std["address"])
df_std[["address", "address_std"]].sample(10, random_state=42)


,address,address_std
33553,245 Myers Union Apt. 852,245 myers union apt. 852
9427,686 Julia Union Apt. 529,686 julia union apt. 529
199,442 Katrina Drives,442 katrina drives
12447,59419 Williams Squares Apt. 528,59419 williams squares apt. 528
39489,5919 Coleman Crest,5919 coleman crest
42724,052 Patel Islands,052 patel islands
10822,5137 Chapman Falls Apt. 415,5137 chapman falls apt. 415
49498,801 Jones Lights,801 jones lights
4144,77227 Eric Isle Apt. 901,77227 eric isle apt. 901
36958,7395 Miller Neck,7395 miller neck


## 5. Country standardization

Sebelum membuat mapping, cek dulu pola aktual variasi penulisan — jangan mengarang daftar sinonim negara tanpa bukti dari data.

In [38]:
print(f"Jumlah unique country (raw): {df_std['country'].nunique()}")
df_std["country"].value_counts().head(30)


Jumlah unique country (raw): 243


country
Korea                                                  416
Congo                                                  393
Palau                                                  245
Switzerland                                            239
United States Virgin Islands                           238
Northern Mariana Islands                               238
Mayotte                                                236
British Indian Ocean Territory (Chagos Archipelago)    235
Belize                                                 234
Botswana                                               234
Cameroon                                               234
Wallis and Futuna                                      234
Sudan                                                  233
Heard Island and McDonald Islands                      233
Latvia                                                 231
Tunisia                                                231
Somalia                                         

In [39]:
# Standardisasi format dasar dulu (lowercase, trim) -- baru dilihat apakah
# cardinality turun signifikan, sebagai bukti bahwa variasinya memang cuma
# masalah format (bukan benar-benar 243 negara berbeda)
df_std["country_std"] = normalize_text_basic(df_std["country"])
print(f"Jumlah unique country SETELAH lowercase+trim: {df_std['country_std'].nunique()}")

# Jika masih tinggi, berarti variasinya bukan sekadar case/whitespace,
# melainkan singkatan/ejaan berbeda (mis. 'USA' vs 'United States') --
# butuh mapping manual yang HARUS dibuat berdasarkan value_counts aktual
# di atas, bukan daftar sinonim generik dari luar.


Jumlah unique country SETELAH lowercase+trim: 243


**Keputusan yang menggantung:** jika cardinality `country_std` masih jauh di atas jumlah negara riil (~195-250), perlu keputusan desain berikutnya: apakah `country` layak jadi supporting field sama sekali, atau didrop dari proses matching karena terlalu noisy. Keputusan ini BELUM diambil di notebook ini — tunggu hasil cell di atas dulu.

## 5b. Parse `dob` menjadi datetime (untuk blocking & comparison)

`dob` masih bertipe `str` format `dd/mm/yyyy`. Cell ini mem-parsing-nya
menjadi datetime dan membuat kolom `dob_std` bertipe `YYYY-MM-DD`
agar blocking key konsisten. `df_raw` tidak diubah.


In [43]:
dob_parsed = pd.to_datetime(df_std["dob"], format="%d/%m/%Y", errors="coerce")
print(f"dob parsed: {dob_parsed.notna().sum():,} / {len(dob_parsed):,} valid")
print(f"dob invalid (NaT): {dob_parsed.isna().sum():,}")

df_std["dob_std"] = dob_parsed.dt.strftime("%Y-%m-%d")
print(f"dob_std unique: {df_std['dob_std'].nunique():,}")
print(f"dob_std missing: {df_std['dob_std'].isna().sum():,}")
print(f"Sample dob_std: {df_std['dob_std'].dropna().head(5).tolist()}")
print(f"df_std shape sekarang: {df_std.shape} | kolom baru: {[c for c in df_std.columns if c not in df_raw.columns]}")

df_std["dob_std"] = normalize_text_basic(df_std["dob"])

# Validasi: semua non-null harus match format YYYY-MM-DD
bad_dob = df_std["dob_std"].dropna()
bad_dob_count = (~bad_dob.str.match(r'^\d{4}-\d{2}-\d{2}$')).sum()
print(f"dob_std: {df_std['dob_std'].notna().sum():,} non-null, format tidak valid: {bad_dob_count}")
df_std[["dob", "dob_std"]].sample(5, random_state=42)


dob parsed: 50,000 / 50,000 valid
dob invalid (NaT): 0
dob_std unique: 17,733
dob_std missing: 0
Sample dob_std: ['1988-04-11', '1955-09-19', '1971-11-02', '1955-03-25', '1968-06-25']
df_std shape sekarang: (50000, 23) | kolom baru: ['first_name_std', 'last_name_std', 'email_std', 'email_is_shared_pattern', 'phone_main_std', 'phone_extension_std', 'address_std', 'country_std', 'dob_std']
dob_std: 50,000 non-null, format tidak valid: 50000


,dob,dob_std
33553,21/10/1971,21/10/1971
9427,16/12/1989,16/12/1989
199,04/07/2005,04/07/2005
12447,13/04/1971,13/04/1971
39489,15/03/1998,15/03/1998


## 6. City standardization

Dipindah dari Notebook 04 ke sini (sesuai arsitektur: semua `_std` dihasilkan di Notebook 02).

In [44]:
df_std["city_std"] = normalize_text_basic(df_std["city"])
print(f"city_std unique: {df_std['city_std'].nunique():,}")
df_std[["city", "city_std"]].sample(5, random_state=42)


city_std unique: 24,534


,city,city_std
33553,Schmidtview,schmidtview
9427,West Brian,west brian
199,North Andrea,north andrea
12447,New Brandonton,new brandonton
39489,Glendaland,glendaland


## 7. Device ID standardization

Kolom `device_id(s)` dipakai sebagai blocking rule di Notebook 04. Nama kolom mengandung
karakter `(` dan `)` yang rawan error di SQL DuckDB — dibuat alias bersih `device_id_std`.

In [45]:
# Tidak ada transformasi konten -- UUID sudah bersih.
# Tujuan utama: alias nama kolom yang aman untuk DuckDB (hindari karakter spesial)
df_std["device_id_std"] = df_std["device_id(s)"].astype(str)
df_std["device_id_std"] = df_std["device_id_std"].mask(df_std["device_id(s)"].isna(), np.nan)
print(f"device_id_std unique: {df_std['device_id_std'].nunique():,}")
print(f"Contoh nilai:")
print(df_std["device_id_std"].dropna().sample(5, random_state=42).tolist())


device_id_std unique: 48,200
Contoh nilai:
['3fbc6171-6d49-4080-a482-c4c99af7e158', '64bd5dca-1628-464d-a433-1a27dc42bfaa', 'bc7bf030-33da-4093-93d8-404ab675f0ef', '24c21a12-7e3a-4a78-a143-e293eb669179', '63a197f7-0092-40e2-a251-d75b81e4efcd']


## 6. Sanity check sebelum menyimpan output

Pastikan `df_raw` masih utuh (Rule 3) dan tidak ada kolom `_std` yang tercampur ke `df_raw`.

In [46]:
assert list(df_raw.columns) == [
    "customer_id", "first_name", "last_name", "email", "phone_number",
    "gender", "dob", "signup_date", "address", "city", "state",
    "country", "device_id(s)", "source"
], "df_raw berubah! Rule 3 (Preserve raw data) dilanggar -- cek ulang kode di atas."

print("OK -- df_raw tidak berubah.")
print(f"df_std sekarang punya {df_std.shape[1]} kolom (raw {df_raw.shape[1]} + kolom _std baru)")
print([c for c in df_std.columns if c not in df_raw.columns])


OK -- df_raw tidak berubah.
df_std sekarang punya 25 kolom (raw 14 + kolom _std baru)
['first_name_std', 'last_name_std', 'email_std', 'email_is_shared_pattern', 'phone_main_std', 'phone_extension_std', 'address_std', 'country_std', 'dob_std', 'city_std', 'device_id_std']


In [47]:
df_std.to_csv(OUTPUT_PATH, index=False)
print(f"Tersimpan: {OUTPUT_PATH} ({df_std.shape[0]:,} baris x {df_std.shape[1]} kolom)")


Tersimpan: C:\Users\User\Downloads\Fix\data\raw\customers_standarized.csv (50,000 baris x 25 kolom)


## Ringkasan Notebook 02 (berdasarkan hasil aktual)

```text
Kolom _std yang dihasilkan   : first_name_std, last_name_std, email_std,
                                email_is_shared_pattern, phone_main_std,
                                phone_extension_std, address_std, country_std,
                                dob_std
                                (9 kolom baru; df_std = 14 raw + 9 = 23 kolom)

Validasi phone extension     : MERAPAT. phone_number raw 10-18 digit ->
                                phone_main_std mayoritas 10-11 digit
                                (33.246 + 5.964 baris). Extension: 29.966 baris.
                                Sisa: 13 digit=7.906 baris (prefix '001'/_code),
                                3-9 digit=2.884 baris (nomor pendek).

Validasi country cardinality : TIDAK TURUN: 243 -> 243 (lowercase+trim).
                                BUKTI: variasi BUKAN case; 243 dalam rentang
                                jumlah negara riil (~195-250) => tidak perlu
                                mapping manual, tetap supporting field.

Cardinality nama             : first_name 5.429->3.309 | last_name 7.228->4.088
Cardinality lain             : email 46.363->46.363 | address 48.200->48.200
Email shared pattern         : 1.550 baris

dob_std                      : 50.000 valid (format dd/mm/yyyy), dob_std n_unique 17.733
Raw integrity                : df_raw tidak berubah (assert lolos, 14 kolom)
Output file                  : customers_standarized.csv (50,000 x 23 kolom)
```

### Open items

1. 2.884 baris 3-9 digit & 7.906 baris 13 digit di phone_main_std -> cek di 03_blocking
2. country_std = lowercase saja (dipakai sebagai supporting field, bukan blocking key)
3. dob sudah ter-parse -> siap untuk blocking rules (phone/dob, email/dob, name/dob)

**Belum dilakukan:** blocking, Splink, threshold, entity clustering.

**Next:** 03_blocking.ipynb baru menghasilkan 1.896 unique candidate pairs (99.9998% reduction) dengan 100% same-customer coverage (1.867/1.867).
